# Forecast Corrections

How to correct erroneous forecast values and read the full revision history.

ClickHouse is append-only — corrections are new forecast rows with a later `knowledge_time`. The overlapping read patterns then automatically surface the corrected values.

In [1]:
try:
    import urllib.request

    import google.colab

    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/rebase-energy/timedb/main/examples/colab_setup.py", "/tmp/colab_setup.py"
    )
    exec(open("/tmp/colab_setup.py").read())
except ImportError:
    pass

In [2]:
from datetime import UTC, datetime, timedelta

import polars as pl
import timedb as td

base_time = datetime(2025, 1, 1, tzinfo=UTC)
valid_times = [base_time + timedelta(hours=i) for i in range(24)]

## Setup

Insert an initial forecast run that contains erroneous values at hours 10–12.

In [3]:
td.delete()
td.create()

td.create_series("wind_forecast", unit="MW", labels={"site": "Gotland"}, overlapping=True)

col = td.get_series("wind_forecast").where(site="Gotland")
kt_run1 = base_time - timedelta(hours=12)

values = [5.0 + i * 0.2 for i in range(24)]
values[10] = 999.0
values[11] = 999.0
values[12] = 999.0

col.insert(
    pl.DataFrame(
        {
            "knowledge_time": pl.Series([kt_run1] * 24).cast(pl.Datetime("us", "UTC")),
            "valid_time": pl.Series(valid_times).cast(pl.Datetime("us", "UTC")),
            "value": values,
        }
    )
)
print("✓ Run 1 inserted (with errors at hours 10–12)")

Creating database schema...
  ✓ PostgreSQL: series
  ✓ ClickHouse: runs, flat, overlapping_short/medium/long
✓ Schema created successfully


✓ Run 1 inserted (with errors at hours 10–12)


## Discover the Error

Read back the initial forecast and inspect the spike at hours 10–12.

In [4]:
ts = col.read()
print(ts)

TimeSeries
┌─────────────────────────────────────────┐
│  Name:             wind_forecast        │
│  Shape:            SIMPLE               │
│  Rows:             24                   │
│  Timezone:         UTC                  │
│  Unit:             MW                   │
│  Timeseries type:  OVERLAPPING          │
│  Labels:           {'site': 'Gotland'}  │
├─────────────────────────────────────────┤
│                    wind_forecast        │
│  2025-01-01 00:00            5.0        │
│  2025-01-01 01:00            5.2        │
│  2025-01-01 02:00            5.4        │
│  ...                         ...        │
│  2025-01-01 21:00            9.2        │
│  2025-01-01 22:00            9.4        │
│  2025-01-01 23:00            9.6        │
└─────────────────────────────────────────┘


## Insert Correction

Insert the corrected values as a new run with a later `knowledge_time`. Only the affected time points need to be re-inserted.

In [5]:
kt_correction = base_time
corrected_hours = [10, 11, 12]

col.insert(
    pl.DataFrame(
        {
            "knowledge_time": pl.Series([kt_correction] * 3).cast(pl.Datetime("us", "UTC")),
            "valid_time": pl.Series([base_time + timedelta(hours=h) for h in corrected_hours]).cast(
                pl.Datetime("us", "UTC")
            ),
            "value": [5.0 + h * 0.2 for h in corrected_hours],
        }
    )
)
print("✓ Correction inserted")

✓ Correction inserted


## Verified Correction

`read()` returns the latest value per `valid_time` — hours 10–12 now show the corrected values.

In [6]:
ts = col.read()
print(ts)

TimeSeries
┌─────────────────────────────────────────┐
│  Name:             wind_forecast        │
│  Shape:            SIMPLE               │
│  Rows:             24                   │
│  Timezone:         UTC                  │
│  Unit:             MW                   │
│  Timeseries type:  OVERLAPPING          │
│  Labels:           {'site': 'Gotland'}  │
├─────────────────────────────────────────┤
│                    wind_forecast        │
│  2025-01-01 00:00            5.0        │
│  2025-01-01 01:00            5.2        │
│  2025-01-01 02:00            5.4        │
│  ...                         ...        │
│  2025-01-01 21:00            9.2        │
│  2025-01-01 22:00            9.4        │
│  2025-01-01 23:00            9.6        │
└─────────────────────────────────────────┘


## Full Revision History

`read(overlapping=True)` returns every (knowledge_time, valid_time) pair — the original erroneous values and the correction side by side.

In [7]:
ts_all = col.read(overlapping=True)
df = ts_all.to_polars().sort(["valid_time", "knowledge_time"])
print(f"✓ {ts_all.num_rows} rows across {df['knowledge_time'].n_unique()} knowledge times")
print(
    df.filter(
        pl.col("valid_time").is_between(
            base_time + timedelta(hours=9),
            base_time + timedelta(hours=13),
        )
    )
)

✓ 27 rows across 2 knowledge times
shape: (8, 3)
┌─────────────────────────┬─────────────────────────┬───────┐
│ knowledge_time          ┆ valid_time              ┆ value │
│ ---                     ┆ ---                     ┆ ---   │
│ datetime[μs, UTC]       ┆ datetime[μs, UTC]       ┆ f64   │
╞═════════════════════════╪═════════════════════════╪═══════╡
│ 2024-12-31 12:00:00 UTC ┆ 2025-01-01 09:00:00 UTC ┆ 6.8   │
│ 2024-12-31 12:00:00 UTC ┆ 2025-01-01 10:00:00 UTC ┆ 999.0 │
│ 2025-01-01 00:00:00 UTC ┆ 2025-01-01 10:00:00 UTC ┆ 7.0   │
│ 2024-12-31 12:00:00 UTC ┆ 2025-01-01 11:00:00 UTC ┆ 999.0 │
│ 2025-01-01 00:00:00 UTC ┆ 2025-01-01 11:00:00 UTC ┆ 7.2   │
│ 2024-12-31 12:00:00 UTC ┆ 2025-01-01 12:00:00 UTC ┆ 999.0 │
│ 2025-01-01 00:00:00 UTC ┆ 2025-01-01 12:00:00 UTC ┆ 7.4   │
│ 2024-12-31 12:00:00 UTC ┆ 2025-01-01 13:00:00 UTC ┆ 7.6   │
└─────────────────────────┴─────────────────────────┴───────┘


## Summary

ClickHouse is immutable — there is no `UPDATE`. A correction is just another insert with a later `knowledge_time`. The overlapping read patterns surface it automatically: `read()` returns the corrected values, `read(overlapping=True)` preserves the full audit trail.

| Step | How |
|------|-----|
| Insert original data | `col.insert(df, knowledge_time=kt_run1)` |
| Correct specific rows | `col.insert(corrections, knowledge_time=kt_later)` |
| Read corrected values | `col.read()` — latest knowledge_time wins |
| Audit original + correction | `col.read(overlapping=True)` |